# Attendance Grading

* Step 1: Input the proper session length into the session_length variable
* Step 2: Input the csv name into the csv_name variable
* Step 3: Input the assignment name from the gradebook into the assignment_name variable
* Step 4: Make sure the zoom recording csv has records occuring after the due date removed before running results (not yet fixed).

Results are retrieved in two csv's "Testing.csv" and "grading_results.csv". The first csv "Testing.csv" returns with more columns joined to the gradebook "Name", "View Duration (minutes)" and "Grade" to verify that the grading script is placing the correct grades in the correct student's row. "Testing.csv" is for debugging/testing purposes not to be put on Canvas.

Final results are placed in the "grading_results.csv" use that for inputting into canvas.

TODO: Removing columns from the zoom recording csv where attendance is inadmissible for credit (too late) automatically rather than having to manually remove them.

In [318]:
import pandas as pd
import re
import datetime

# Set the variable session_length to how long the zoom session was for to accurately get grading results (in minutes)
session_length = 158

# Name of attendance csv (change as needed)
csv_name = "zoomus_recordinganalytics_01-13-2025-play.csv"

# Name of assignment in the gradebook
assignment_name = "1/13/25 Attendance (2257194)"

# Gradebook name
gradebook_name = "2025-01-27T2126_Grades-COT5930_012_16128.csv"

# Date where attendance viewing is no longer counted. Format in mm-dd-yyyy
# cutoff_date = "01-06-2025"

In [319]:
# # Find the date info from the recording name and extract the month date and year from the csv filename

# mat = re.findall("[0-9]", csv_name)

# month_date_year = []
# for i in range(0, len(mat) - 4, 2):
#   month_date_year.append(''.join(mat[i:i+2]))
#   # mat = ''.join(mat[:2])
# month_date_year.append(''.join(mat[-4:]))

# # Input date time parameters
# beginning_month = int(month_date_year[0][1:])
# beginning_date = int(month_date_year[1][1:])
# beginning_year = int(month_date_year[2])

# # Input the end date parameters here
# end_month = 1
# end_date = 14
# end_year = 2025

# beginning_period = datetime.datetime(beginning_year, beginning_month, beginning_date)
# end_period = datetime.datetime(end_year, end_month, end_date)

# print(end_period)
# print(beginning_period)



In [320]:
# def create_datetime(end_period, row_date):



#   dt = datetime.datetime.strptime(row_date, "%y%y")
#   print(dt)
#   # month_date_year = []
#   # for i in range(0, len(mat) - 4, 2):
#   #   month_date_year.append(''.join(mat[i:i+2]))
#   # # mat = ''.join(mat[:2])
#   # month_date_year.append(''.join(mat[-4:]))

#   # # Input date time parameters
#   # month = int(month_date_year[0][1:])
#   # date = int(month_date_year[1][1:])
#   # year = int(month_date_year[2])


#   dt = datetime.datetime(year, month, date)

#   return dt <= end_period

In [ ]:
# Read initial zoom csv recording data
recording_df = pd.read_csv(csv_name)

# Changing "< 1" in the View Duration (minutes) column to None so the column can be summed
recording_df['View Duration (minutes)'] = recording_df['View Duration (minutes)'].apply(lambda x: None if x == '< 1' else int(x))

# Group by student names and emails and sum the view duration in minutes for every individual student
recording_df = recording_df.groupby(["Name", "Email"], as_index=False)['View Duration (minutes)'].sum()
recording_df



In [ ]:
# Changing view duration column to int dtype
int_dict = {'View Duration (minutes)': int}
recording_df = recording_df.astype(int_dict)
print(recording_df.dtypes)

In [323]:
# Implementing grading scale function

def grading_scale(session_length):

  ninety_percent_watch = round(session_length * 0.9)
  eighty_percent_watch = round(session_length * 0.8)
  sixty_percent_watch = round(session_length * 0.6)
  forty_percent_watch = round(session_length * 0.4)
  thirty_percent_watch = round(session_length * 0.3)

  recording_df['Grade'] = 'NA'
  recording_df.loc[(recording_df['View Duration (minutes)'] < thirty_percent_watch), 'Grade'] = 0

  recording_df.loc[(recording_df['View Duration (minutes)'] >= thirty_percent_watch) & (recording_df['View Duration (minutes)'] < forty_percent_watch), 'Grade'] = 1

  recording_df.loc[(recording_df['View Duration (minutes)'] >= forty_percent_watch) & (recording_df['View Duration (minutes)'] < sixty_percent_watch), 'Grade'] = 2

  recording_df.loc[(recording_df['View Duration (minutes)'] >= sixty_percent_watch) & (recording_df['View Duration (minutes)'] < eighty_percent_watch), 'Grade'] = 3

  recording_df.loc[(recording_df['View Duration (minutes)'] >= eighty_percent_watch) & (recording_df['View Duration (minutes)'] < ninety_percent_watch), 'Grade'] = 4

  recording_df.loc[(recording_df['View Duration (minutes)'] >= ninety_percent_watch), 'Grade'] = 5

In [324]:
# Call grading scale with session length of recording csv being analyzed
grading_scale(session_length)

# Used in testing whether the grades were accurately applied to students in the gradebook
recording_df.to_csv("Preliminary.csv", index=False)

In [ ]:
# Read in the course gradebook
course_gradebook_df = pd.read_csv(gradebook_name)

# Change zoom recording column name from email to SIS Login ID for joining with gradebook csv
recording_df = recording_df.rename(columns={"Email": "SIS Login ID"})

recording_df_final = recording_df.copy()

# Left join the gradebook with the recording df
merged_df_experimental = pd.merge(course_gradebook_df, recording_df, on="SIS Login ID", how="left")
merged_df_final = pd.merge(course_gradebook_df, recording_df_final, on="SIS Login ID", how="left")

# Move Attendance grades into appropriate column at appropriate rows
merged_df_experimental.loc[2:, assignment_name] = merged_df_experimental.loc[2:, "Grade"]
merged_df_final.loc[2:, assignment_name] = merged_df_final.loc[2:, "Grade"]
merged_df_final = merged_df_final.drop(["Name", "View Duration (minutes)", "Grade"], axis=1)

merged_df_experimental = merged_df_experimental.fillna(value = {assignment_name: 0})
merged_df_final = merged_df_final.fillna(value = {assignment_name: 0})

# CSV where recording time and names and grades are included to verify the script grades properly
merged_df_experimental.to_csv("Testing.csv", index=False)

# Final Gradebook
merged_df_final.to_csv('grading_results.csv', index=False)



In [309]:
# Makes a separate csv with grade counts
grade_count = recording_df.copy()
grade_count = grade_count.groupby(['Grade'])['Grade'].size()
grade_count.to_csv('grade_count.csv')